# Paper 3: unstratified target-prevalence evaluation

**New post hoc experiment — CPU runtime.** Run this notebook in Google Colab using **Runtime → Run all**. Allow Drive access when prompted. No GPU is needed. The original files are read only.

This samples new panels from all four raw benchmark captures without balancing the classes: 12 transfer directions × five seeds × three saved classifiers = 180 jobs. Models, preprocessing and thresholds stay fixed. Exact 49-feature overlaps with all four source partitions are removed before prediction. Panels contain 100,000 eligible rows; the actual observed class proportions are reported. A panel that cannot meet that size from the fixed 200,000-row pool stops the run for inspection.

These are **benchmark capture proportions, not production prevalence**. This does not add a new independent dataset, perform attack-family-held-out training, refit models or finish the other remaining reviewer requests. It does address evaluation beyond the original balanced target panels.

## Before starting

Your existing project must remain at:
`/content/drive/MyDrive/research/researchdata/NIDS_Research/Paper3_DataSet`

It needs the four original raw ZIPs, `Paper3_Sanitized`, `Paper3_Source_Models` and the feature schema. The notebook contains 245 expected file checksums from your verified earlier reports. It stops if historical inputs differ, or if any of the 60 source-model replays fails. Do not edit the expected checksums to bypass a failure.

The full raw scan covers about 66.9 million rows, followed by 180 model evaluations. Keep the Colab session connected and let the printed progress advance. It can take substantial CPU time; a duration has not been measured for this new experiment. New work happens on local Colab disk; completed stages are copied to Drive.

## Results and interruptions

Results are saved in:
`Paper3_DataSet/supplementary_natural_prevalence_20261024/run_v1`

At the end a small **`natural_prevalence_report_....zip`** downloads. Upload that ZIP in this conversation. It contains summary tables, audit records and the code, but no flow-level data or model binaries. **PASS** means all 180 jobs completed; **INCOMPLETE** is an error report, not a successful experiment.

If Drive disconnects, keep the current runtime. Rerun the mounting cell, then the final run cell. If the runtime itself was lost, run all cells again. Matching complete stages resume; an interrupted raw scan restarts that target. Do not delete or alter checkpoint files. The final cell can still make a local error ZIP if Drive fails.

Software installation occurs before the experiment imports its dependencies. If Colab asks you to restart the runtime, restart it and then use Run all again.


In [ ]:
# Install the versions used in your successfully replayed source-model workflow.
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','-q',
 'numpy==2.1.3','pandas==2.2.3','scipy==1.16.3','scikit-learn==1.6.1',
 'xgboost==3.4.1','joblib==1.6.0','pyarrow==21.0.0'])
print('Dependencies installed. Restart only if Colab requests it.')


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive',force_remount=True)
PAPER3_ROOT=Path('/content/drive/MyDrive/research/researchdata/NIDS_Research/Paper3_DataSet')
if not (PAPER3_ROOT/'Paper3_Materialized/feature_schema_v1.json').is_file():
    raise FileNotFoundError('Project/schema not found at '+str(PAPER3_ROOT))
print('Project found:',PAPER3_ROOT)


## Fixed input records
The metadata below comes from the successful full temporal replay, the original input-readiness audit and your completed raw-metadata recovery. Raw-archive hashes are references established during that recovery, not claims of historical pre-label seals. The frozen plan is embedded in the next cell and saved before new sampling starts.


In [ ]:
PINNED_INPUTS = {
  "NF-BoT-IoT-v3.zip": "cd0d3eab5b2da73a377561492aafae078432e69c7932b6302a9fbd7f51429eca",
  "NF-CSE-CIC-IDS2018-v3.zip": "7399bdef73d29875ab388a53e2ae7e592d1952c4ef07ae6de195ba133823d97f",
  "NF-ToN-IoT-v3.zip": "b3b6256e970a8986d87716edea4bfd436d68f04058bf5c26cf67e6d839c83698",
  "NF-UNSW-NB15-v3.zip": "6d51554df43e324ba8d68fc31a02a83d860ba0a3a93d63320a4a6fc314c9cdfd",
  "Paper3_Materialized/feature_schema_v1.json": "67d61eebb3377e3053f943f1e4c031237b3ef35dd06296d8538fe552b16be0c6",
  "Paper3_Sanitized/NF_BoT_IoT_v3/seed_2027/calibration.parquet": "628a07c66ee41949f469ddfdb0c5ad0d48ef867a37b3fbd07d876e8bbdc73ce5",
  "Paper3_Sanitized/NF_BoT_IoT_v3/seed_2027/environment.parquet": "9038da7de0a881e2d8b1468709e43556aa44c6170b80b71a748579d0e6abb018",
  "Paper3_Sanitized/NF_BoT_IoT_v3/seed_2027/source_test.parquet": "2b0bc7cc666945574a29aa13f97803fb587e4c7066af44350508c840f08de1ce",
  "Paper3_Sanitized/NF_BoT_IoT_v3/seed_2027/train.parquet": "14a818f65a40ad78b82cd72aea23e7df85b53681af0760ece1d549bf2de09af9",
  "Paper3_Sanitized/NF_BoT_IoT_v3/seed_2028/calibration.parquet": "0495a48fba16e3a5c2b64949ce850523e6a11b4b968f35bb447fefef78aaf1d0",
  "Paper3_Sanitized/NF_BoT_IoT_v3/seed_2028/environment.parquet": "356f38ea7bdf9bedb1b0e07d2c6c07f7a4467b44c28e64f8839c38dd5fe4f009",
  "Paper3_Sanitized/NF_BoT_IoT_v3/seed_2028/source_test.parquet": "49de37f151f2cda111398b8bf75ce7e9d59d26276ad44f66ca73ec000d83f74d",
  "Paper3_Sanitized/NF_BoT_IoT_v3/seed_2028/train.parquet": "c59ab8ae48e7177a35e16c9fe12ecba706a480322f9a7077b470b470260cd232",
  "Paper3_Sanitized/NF_BoT_IoT_v3/seed_2029/calibration.parquet": "e662915a0340d241ddc91f5347a937ba5817e4e95a4eec18083eb408df395739",
  "Paper3_Sanitized/NF_BoT_IoT_v3/seed_2029/environment.parquet": "a4853ec77d91ec27ce18b788f75969c617a38cf424f1de8a173574693783a2cf",
  "Paper3_Sanitized/NF_BoT_IoT_v3/seed_2029/source_test.parquet": "7fff718f64a3d216e0b00d5d2e76ef9a98edc480661de9c894ead197ae4e1ea6",
  "Paper3_Sanitized/NF_BoT_IoT_v3/seed_2029/train.parquet": "d0cc1d0efa9b7b37fb2cd2d6df1fbfbf7def5af8144a6b6e6198c96d7597d0d3",
  "Paper3_Sanitized/NF_BoT_IoT_v3/seed_2030/calibration.parquet": "2266fcc8e6b125ff9a5ee1334f45d1dc22adc4dd303548a2ce1e5cc9782af325",
  "Paper3_Sanitized/NF_BoT_IoT_v3/seed_2030/environment.parquet": "d225a8a8a54e15aad60c5ce6b7dd4a1a6a420ac9c2d24f6513a656238e448e0c",
  "Paper3_Sanitized/NF_BoT_IoT_v3/seed_2030/source_test.parquet": "3dc968ed856c70e5569949dcb7690bdb2c488b0581063d4b6d033ff4a7b1f11a",
  "Paper3_Sanitized/NF_BoT_IoT_v3/seed_2030/train.parquet": "b5bf843f2a50eb9c35d75a1d2ea6842d27a532652372d880142664b1fa196c27",
  "Paper3_Sanitized/NF_BoT_IoT_v3/seed_2031/calibration.parquet": "3e947cae3cdf051ba2e5453830348c390522fa316fd7792301909b3d61f54904",
  "Paper3_Sanitized/NF_BoT_IoT_v3/seed_2031/environment.parquet": "48773ddccb6ba3fff2d80b0fb2672bb2cfcfae4f63646d5e097f8d135a52a4c1",
  "Paper3_Sanitized/NF_BoT_IoT_v3/seed_2031/source_test.parquet": "f198f84872382e70f1044bcdcad9e3ae3bf2d6f633343b2ab413ec0c2b4fe2a7",
  "Paper3_Sanitized/NF_BoT_IoT_v3/seed_2031/train.parquet": "9f9a950c33a20f7306c01e673e0df1e2252cd92ea2b5d1f6c741f13d485dc79b",
  "Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2027/calibration.parquet": "2885e67e88dde9d5488c94af486b2bf1d13d1d3cb090b481f116960c05db8c85",
  "Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2027/environment.parquet": "c71f732a117d1c8271018fae8e70469365098cf8358f66c361dbe0126c8acc56",
  "Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2027/source_test.parquet": "12200066b125db69e51fc58043543942162a591834a38e0a69d9e26d42d2bb07",
  "Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2027/train.parquet": "27ce500af0b403948b55fda28ea87fa9215a196efad4c710e99fa4643e873736",
  "Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2028/calibration.parquet": "931b742549a2d9980c938d591a24282706069c8ac90e298828244dcec9d00866",
  "Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2028/environment.parquet": "40102921b7b7c9009ff286ecaeca85a34f444bcd088d56ab7f73fb714b483ac0",
  "Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2028/source_test.parquet": "9c4c427f26b2a2c53ef5d447ad2515d59b1acdacbd38cb7b936dd8e1a6e0b292",
  "Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2028/train.parquet": "ddba8ea1bb07812b3fd43b506992e7ec537b4941f7726faec0a41f666a8b9f93",
  "Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2029/calibration.parquet": "81296a4e962dc565b95183c5e9c674bc5beb25f88a54ef9ffab00f2730090b0b",
  "Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2029/environment.parquet": "aca54a99d2050dc0a73a7a6692fb8ac655eb22d9d7a0d41262b4ba81794212ba",
  "Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2029/source_test.parquet": "ea8de15a143e2c2d467d984c24a934c2ee8c0a71f8bf9048437fe7d35472137a",
  "Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2029/train.parquet": "84ec860b817718b14417af7aefbe8dd53fb9180aa9945f7d576404a94dc609d4",
  "Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2030/calibration.parquet": "7b4f25de62976590ac5cb1397f774f96f7706254fc42cf908807cf313ef00787",
  "Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2030/environment.parquet": "addc3fb71a67fa136c6b5dc9d9bede4eff60722d28d3d416425271377cd03a86",
  "Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2030/source_test.parquet": "3b20af47e8fabbc1b6d4c575eee52309ac71ca3be49302fe1d6bf1285b7dc515",
  "Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2030/train.parquet": "61a82c0e5875db2b0a3765c2061dd391b68e32133eec447ed1b8ad626ccd365c",
  "Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2031/calibration.parquet": "6f3921c5882712e8a7f1afcbea6051d968ef7f575c0501d0db96bec8f2520fd5",
  "Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2031/environment.parquet": "9adcfbf72720505c323faea3c1a91c20520dd0bb888ce1203a29b1d55bc94042",
  "Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2031/source_test.parquet": "86345d53c306ad04e39285c811fb04d8d8b97130ee7dd61d6d00ac465882d742",
  "Paper3_Sanitized/NF_CSE_CIC_IDS2018_v3/seed_2031/train.parquet": "efdf6b1ad2a081baedb8eb1c663ab19c99b13e819d0c8addcac06738b4fa94cc",
  "Paper3_Sanitized/NF_ToN_IoT_v3/seed_2027/calibration.parquet": "b0c631594763c8f65787785f9ea4966bfb81c32f07dcebd93a6be82c533cfa32",
  "Paper3_Sanitized/NF_ToN_IoT_v3/seed_2027/environment.parquet": "8a7dd21d970df7248f01e723c2b4a125903b5431c6aebf2b40522ab8c60298dd",
  "Paper3_Sanitized/NF_ToN_IoT_v3/seed_2027/source_test.parquet": "d95e3fb3c7d0f9a256736a09f6565114786aa34178934c4dc895a8a1d37c993c",
  "Paper3_Sanitized/NF_ToN_IoT_v3/seed_2027/train.parquet": "5535a60be84cd758ca9362e7045246aaa1face3054df80e9a4a7325d003eec5b",
  "Paper3_Sanitized/NF_ToN_IoT_v3/seed_2028/calibration.parquet": "5e042bad2ac3da65b876b4ffcb132be46b2f93113f7da9e2affec9bd313ffc4d",
  "Paper3_Sanitized/NF_ToN_IoT_v3/seed_2028/environment.parquet": "fed6b48b4792fcde4bf2772f22c1a34a8b0083a2851d53575f3baa7f15b13336",
  "Paper3_Sanitized/NF_ToN_IoT_v3/seed_2028/source_test.parquet": "e81625480f3fd997b4fc4a52722786922db386fb84b2f5d4fefa947d83ac2e1f",
  "Paper3_Sanitized/NF_ToN_IoT_v3/seed_2028/train.parquet": "5b61026d0ca14abb02534d29758999e569a7dda65413a2974a4ab21a4e1f2aee",
  "Paper3_Sanitized/NF_ToN_IoT_v3/seed_2029/calibration.parquet": "2e76b4fcf60391167278eabe2e226e39f1d1a14e39443ee2313313bd213d20be",
  "Paper3_Sanitized/NF_ToN_IoT_v3/seed_2029/environment.parquet": "8ca2a80cd2649066ca40ade7c6ca1c666c39232834d0a4459a41c913c1539fc2",
  "Paper3_Sanitized/NF_ToN_IoT_v3/seed_2029/source_test.parquet": "041e23affe790efd8d58587047184c81b85074f5cfe57a0754398116c45b5af1",
  "Paper3_Sanitized/NF_ToN_IoT_v3/seed_2029/train.parquet": "1c27b7b6c76c58ff8ee663247b846c7a72f2ba36da30580af7c504ef29b4641e",
  "Paper3_Sanitized/NF_ToN_IoT_v3/seed_2030/calibration.parquet": "bd5d8b3f3026062f0d076179ff1bcf02dc1df907aae401de36e47ee0cff1569b",
  "Paper3_Sanitized/NF_ToN_IoT_v3/seed_2030/environment.parquet": "e9437952fce5264ab0a176316511733a5d69b94f1d2b336368ab128201fbbf41",
  "Paper3_Sanitized/NF_ToN_IoT_v3/seed_2030/source_test.parquet": "fac85d8b954af6b20b529feef19e9f84569c7884321cff78c06ad8e19c84cdfa",
  "Paper3_Sanitized/NF_ToN_IoT_v3/seed_2030/train.parquet": "b71b51764b0ce814d8b831a9241637896e2055009c77975849c830edfd3904f2",
  "Paper3_Sanitized/NF_ToN_IoT_v3/seed_2031/calibration.parquet": "64d1555d8f9e5f22f22cc43cdfa740cef8956a9474ce0e590137cce7e4c0f0e6",
  "Paper3_Sanitized/NF_ToN_IoT_v3/seed_2031/environment.parquet": "ed121e11412e3bcf4a732398427db4008d187eeacc7296df3f19c6d1afbaad68",
  "Paper3_Sanitized/NF_ToN_IoT_v3/seed_2031/source_test.parquet": "3487e8b958fb92e288ce62ebf3979ed63da7efadb765026da9c88d6fb8e31e5f",
  "Paper3_Sanitized/NF_ToN_IoT_v3/seed_2031/train.parquet": "8db5c88c5f4dc288cb3674987c605a99816a122303a22b33eab13c0915c199bd",
  "Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2027/calibration.parquet": "e30e4d663acb8c82d94f48503e20fc6dcf888c13a1b7faef1707b3f8854af32b",
  "Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2027/environment.parquet": "7f7f6f2663a991da3ddbf383eed9097bdfbe86ae139c162a40ddba07665cf961",
  "Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2027/source_test.parquet": "d7bd279fa1a0ddac02d9397fd68abdea33737f7204ec5dd7e567c243d65f44c8",
  "Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2027/train.parquet": "f732bfba933d1256652544886813c020ee2309cac9181bada34674dcf5626119",
  "Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2028/calibration.parquet": "4e4deba40108efe46df0785c2d40a0d7a45cab50c6c40e01826bda08743b1cee",
  "Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2028/environment.parquet": "20d4cf015e1cb7d8c99ad280bb97c407d333326e09af5e9848fcb6a42494418c",
  "Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2028/source_test.parquet": "96bab17b4847200c038c3e7c78f408558721f0842a06cf71cdcc26c1217b7a55",
  "Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2028/train.parquet": "5ffa313db0ffd0e01c122cc60397253079f5cecabf355ce48610222410d0764d",
  "Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2029/calibration.parquet": "e1b7605bfe97c28f49f228b19ccf5c92724c88e951905c4a5e6510cca9c0f773",
  "Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2029/environment.parquet": "c5d8913198e859ec448fff0f1538bdf62e809f9f85c9a9678d6a72bf8b36fedc",
  "Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2029/source_test.parquet": "5ef2fa26656225c83c6ab8a91d3d6dd9b2dfbe97228a760a35e9221042364150",
  "Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2029/train.parquet": "d69b09bc9a9faf8dde11b233e208579f1204412328baa9e0a32a9bf15c107fb7",
  "Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2030/calibration.parquet": "a19de0f4fc3db885b282424d5e42a247828f008479886a9e134263c6c354c6ed",
  "Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2030/environment.parquet": "948f46c79ff4b22e08055287d4e23f4781ccedf06b804e333406a769b059b546",
  "Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2030/source_test.parquet": "f1a17d74421e102eb0cf4a214949147ad64af206df4b52f8a528e3181578c9a8",
  "Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2030/train.parquet": "52b7f7c6907338770042bdc7605b2e5c59e6143716eaae8454b67c8d8c683a7b",
  "Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2031/calibration.parquet": "42b84079769948fac863ffb2504851b299bd7b404495310be3aa6a4f24137250",
  "Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2031/environment.parquet": "a2dd1b819ccedf317b42159a1bb5268869738821394eb9bc86e73ab5cac227be",
  "Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2031/source_test.parquet": "c9ff9f2791f5f1aa027c6ba44c95d74e1613c8649e1076e0064eaed033d7819c",
  "Paper3_Sanitized/NF_UNSW_NB15_v3/seed_2031/train.parquet": "867abf09516e6bd6814912870be39da8c5bd4261d538c9cba756aa0440ced14a",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2027/LogisticRegression_model.joblib": "945eff37dccb31c89f5af1e716de8dc251e51b1162bec3434821785d519b4d63",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2027/LogisticRegression_source_test_predictions.parquet": "35beeda579e443f9c2bb7d76b53f16fa2129be9e9d8b7448b0d1c0c39c3988a3",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2027/RandomForest_model.joblib": "35091e1a0f65016d10b267bc4343695d74272a8e13d4a57924e943dc49019038",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2027/RandomForest_source_test_predictions.parquet": "7997e8922079338b077b950b67d51669c5035db980f0e3a29108af2737b23fb0",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2027/XGBoost_model.joblib": "e4da21653769fc8c3de80d80e22cdf54d519054c233aae48bd46a789fceb21b0",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2027/XGBoost_source_test_predictions.parquet": "11331651558e20a2d788c7d88f4a2ea66eb616989893d41007e0594d72861d59",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2027/source_preprocessor.joblib": "bb4b521c0c4fa3558010955f594e2454e447cbeeaa71c7b8f44c46ce3f0d0ccd",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2027/source_seed_complete.json": "ace21e8add5de022fcc8d73cf1cf4c9cda5010edc7d32268e9b0d9cce0df27b4",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2028/LogisticRegression_model.joblib": "37fe2ab4ed025a0ab3baea97ccbbd677ead47da05144f064a69e2315888d0c3a",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2028/LogisticRegression_source_test_predictions.parquet": "591311a34e5fc433afd05c5935283cf7b72a725aa30fded0c377b912f62ad9e9",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2028/RandomForest_model.joblib": "353c25340ef656b47574f02dcf27e2775a3fc08e0a4a65f967bae7aaa3bea552",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2028/RandomForest_source_test_predictions.parquet": "053f358fd3d3d191197dd24adbd53d7deaee75e5b578768abf2cf6ce3b42f8b1",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2028/XGBoost_model.joblib": "3f25abce8127346480040c82de421ed0b5e836ab8bff6401e95fd45a196fcfc1",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2028/XGBoost_source_test_predictions.parquet": "6f324273721dff057850ee4a6c69a9b2294d5d7369262bf3b3f8d9a6a1beba4c",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2028/source_preprocessor.joblib": "b88b8d679c0314af1f22ddc47de2837339714df5ba1e150c331ab84d019df359",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2028/source_seed_complete.json": "4185865c67410c53a69cc8ba5f54d6fc1af19235404cbfb364e499358234b246",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2029/LogisticRegression_model.joblib": "1a55641b16bd3a793e37502da491ed5f1b8f7eddada3200f9bbd6b4a8c33f9ed",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2029/LogisticRegression_source_test_predictions.parquet": "4d4ce807e220c58edc61ded8ee013dc88c6bb552e9611d14abebe1ca23402a94",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2029/RandomForest_model.joblib": "113f2c0ea1f2d9936615219c304b220c3b59dc9d30bdae541e1644135569e2a7",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2029/RandomForest_source_test_predictions.parquet": "58ddce4fdbc625de667762fb617d913f2196dfb08e0ddad077796607b8b64252",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2029/XGBoost_model.joblib": "d621e20a0047102e6beadcf2359c73957d0d045518a1f211d6378e3b2a3d120c",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2029/XGBoost_source_test_predictions.parquet": "c1ff3e41651f00cf7fb8fb18d2da6399c2bec65436183a715228b2e7dc31883c",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2029/source_preprocessor.joblib": "a976b2bb69d68453e03a1ec9dc31c6104d81e01d775d37858d427888af876bf7",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2029/source_seed_complete.json": "6d80e270704e7a864b7656c917f2dbc19aa7b6f3f75b53ca346ef0b2000f94d1",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2030/LogisticRegression_model.joblib": "cd5826138e28c8cc60837649687452334dd9abc1c07f7b613c361740b53e323c",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2030/LogisticRegression_source_test_predictions.parquet": "f4e07f181619b663f79edf5bfbdad1868a94eba044bcc8cac547cb102d7db4bc",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2030/RandomForest_model.joblib": "72a9e4159843a11c28cf3d2a5e707185ea96b1825382fecf60fdd64d605d9d2b",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2030/RandomForest_source_test_predictions.parquet": "c0a39ef89d5c188f0d3ed42fb93f45e3db34a475c7182fb91c85b1af114b83c5",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2030/XGBoost_model.joblib": "1a8652ca48ff0584afaee0070ba6ee97dfe4ce41868d07f239f597fef6f405e1",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2030/XGBoost_source_test_predictions.parquet": "3aec7db9833c86fb0edfdb98f146ce07209f01f0d569f793f02366bf5a54a387",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2030/source_preprocessor.joblib": "7a904e921e694947655c0c895a2501e713681575d72381ba069253e3c2ea0c63",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2030/source_seed_complete.json": "c5fc61fe99bfc567ec36c4cc75d2fb012522fd476d9c63de560c5e9218d59975",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2031/LogisticRegression_model.joblib": "c023f8bcf7e919caa20a6331dac32813c3382eea6a7d8d001881a86017228664",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2031/LogisticRegression_source_test_predictions.parquet": "1817849976f516a5ea17ac6c854945f50b967bc78791b2f2a5bb7155ec51f139",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2031/RandomForest_model.joblib": "b41b0e84d3703a2ba76c68e8582dd93a4b04b8a68caf314e44f0671b3d748e53",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2031/RandomForest_source_test_predictions.parquet": "e02bcbecba07fa28a601d71e479e47deec7f73c67d57c46aeaabbf6cdbcbc6bc",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2031/XGBoost_model.joblib": "5cee9d63e801d752ed41123779ae52e1cae66fedfeea39e7c14084fa8e60dfa8",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2031/XGBoost_source_test_predictions.parquet": "79a27b7d350cd00e39071421ffb5f26eb09e291e34f4ac1cab8e196a69e112ec",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2031/source_preprocessor.joblib": "90a5add57a5f9f8f32ce770e89a6944ca6587f15e27f5b680ae5a05114e686ef",
  "Paper3_Source_Models/NF_BoT_IoT_v3/seed_2031/source_seed_complete.json": "03eeabd0b9921cf04b9a7e0c704f8abcdf780d10426c4c44cf91932b558cb8e4",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2027/LogisticRegression_model.joblib": "a4f5f9eeebd98df95b0126285e1e51340d1f4c540b0291e9e40920562f61e731",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2027/LogisticRegression_source_test_predictions.parquet": "5e360a92d62676caff0415312977c84e9c6f192feb457a346d6e2f125800d4fa",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2027/RandomForest_model.joblib": "4910110ddf11f388cef84529e9f4a0beb4840d5b86e78b03c32b3f93764a78b5",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2027/RandomForest_source_test_predictions.parquet": "4694dcb7cc3eb86dc8b34a11a228f66952a53e9a042e03aad7df7a33704d2161",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2027/XGBoost_model.joblib": "27e79a0c9635c86cdd644acef41d2219e7bcc6ad13a239809f3ea2df6034889d",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2027/XGBoost_source_test_predictions.parquet": "752f4f62c74a735066b9ab6e4cfd9460c3efb4895dfaf2298550e2c5deb40016",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2027/source_preprocessor.joblib": "a4430b63923592e6dbede12e73661fe3bfbf6e91aa696ded5e557d84078b4af5",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2027/source_seed_complete.json": "8850dd6839fd53a1d83be2be217d07be8abba2b1fc281c190fb16a0e0ddc3306",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2028/LogisticRegression_model.joblib": "5158c7538c01692a3b01af82d55563cdd5311be203443c69d42688f3bb1d14f8",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2028/LogisticRegression_source_test_predictions.parquet": "95b5abb1d1a9de18d9a7ef7dac4988c2dcd624589e01ba3c990d1fd504bf5b5e",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2028/RandomForest_model.joblib": "7a3f03d67c9cc2040cd23eca68b8ca4c83f1429689c74ae0141614f1d0826003",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2028/RandomForest_source_test_predictions.parquet": "c08923dd74b0916d2e45c194df8fde45be316e1f989748183d0d3d53d0cb171a",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2028/XGBoost_model.joblib": "fd28059501c2386104073399027675abf5d671ff08b877e82083584bd7846c46",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2028/XGBoost_source_test_predictions.parquet": "7328e2484aade588847ea995cc7161b47d809ca9e202cdec407ae86301857661",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2028/source_preprocessor.joblib": "53eab7ca7ab5f77af4cc28a5669bae38254b0c8579dbcd5f3876ec4c3ceea46d",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2028/source_seed_complete.json": "4d8d5e6a60ed147d187b64f7b997382686a1c05769da33724627ecc30e1a7f61",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2029/LogisticRegression_model.joblib": "bdcff4120cc1decba2027880b9e23076a42e6ee409d2c94ce72ebe05c8f840d7",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2029/LogisticRegression_source_test_predictions.parquet": "0daf227d3e5f7a87610c4c960975a4aa9eb7b6bef8048596a53e4182b6fd3c64",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2029/RandomForest_model.joblib": "5bb2a82b08f8779be40ce3d2744df5301a798c59799ffc77d79b0971c8b8ced5",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2029/RandomForest_source_test_predictions.parquet": "4cc87ce446d68274316568da6c26887913bd874ce63d40c262c92a89f71413ed",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2029/XGBoost_model.joblib": "3bb66f3208959b8f6f76bff7821dab2c32cc5103550f3af91649488b44ae62ad",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2029/XGBoost_source_test_predictions.parquet": "0e52dda41e76cd920645c4304b13ce0c5fb720c92520b83f801e4d66e21126fc",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2029/source_preprocessor.joblib": "5a186d8d7823be6589ae0af6b587faa1c901585c841908bff6367d0f7f73267d",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2029/source_seed_complete.json": "676a4b6c7be4418c7e3581c30a8a7057e9ce91d4ceaf3c4d2c666435a2544944",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2030/LogisticRegression_model.joblib": "21f1eb21962f0b449841533f2b75ec310d0cd03f110254d0b0077a6fd3979336",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2030/LogisticRegression_source_test_predictions.parquet": "465e54c09d38729faf042e13e06f93d388ff102c415334977c05247418d5938c",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2030/RandomForest_model.joblib": "f162952b80c4968d7e3ada1f614f65a9d4f72319126628f4478e8afad342931a",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2030/RandomForest_source_test_predictions.parquet": "b72908f3862e4f42e687ae51f989972524ef0b0c1495fad74f5ac6cba09fde76",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2030/XGBoost_model.joblib": "38e70ca3f408d03050eaf2c5a3a9ce3033b4f5d4df717d8ec2141f33ddd7b3f0",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2030/XGBoost_source_test_predictions.parquet": "c8db129da3bae08ba9e044dac3033e94c6195788856548e5b6541aef9ffa3ad5",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2030/source_preprocessor.joblib": "5eeabab67a8c74a9d9aaa1b34f19e4fd23b2f56806a550610e9b4f9bb739982f",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2030/source_seed_complete.json": "22aed016730b35036e44d6076c9fa8721c8a8c3d6cb56292797e0a2ef35eaa87",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2031/LogisticRegression_model.joblib": "c875c70941c212657877e17702c221e09903542aae3d7d3a9818df484ecfc911",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2031/LogisticRegression_source_test_predictions.parquet": "054ae4afde289c20ef0f14ac7c2c98ed9a503cc092658af48b87c393928afa70",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2031/RandomForest_model.joblib": "35ed86f51d563151095ee4b1ed6b4b856fe3197b93e6d6205acea9c28a363595",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2031/RandomForest_source_test_predictions.parquet": "a01b761cfb963eff8afbae811a82a046d9673f6525c4ef2ab81dc0279ab10353",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2031/XGBoost_model.joblib": "0a4e103281f1c550db544fe38076237ee31c64dd7629c1b3012cd8a4ee4c82f5",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2031/XGBoost_source_test_predictions.parquet": "fb8a37b1a4af7fd8b0ee49976e892777104a1d242059e5288a222972c26d8950",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2031/source_preprocessor.joblib": "d1bbae047e6de2404812fc3d4de9c542521027846c33e6f65365aad230323154",
  "Paper3_Source_Models/NF_CSE_CIC_IDS2018_v3/seed_2031/source_seed_complete.json": "1038db5cc8d674200b72779fc63a8a82e38d8cf0499ed24c6bd8ee54e9c7aa05",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2027/LogisticRegression_model.joblib": "a7fb4eb4aa36cb60a88f0b11c7bcd0ad7f9b6afaeb95a81677526b9eb0c0cc2f",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2027/LogisticRegression_source_test_predictions.parquet": "11248c88fdc12d788c73fc076ce951249709db41c208913a9f325bf42afb9052",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2027/RandomForest_model.joblib": "01ae3c4dddf9191e09fdf08f5abc9a1b23a5063eecda240af1a1c8ef7a9e4c2b",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2027/RandomForest_source_test_predictions.parquet": "4f1d3b66e92c33efa0981e3b4d91cc6f656921aac88f2758e792050b735cc6f5",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2027/XGBoost_model.joblib": "4f7283ec182b714a0915aa2afa9d8035b6e801c14bf07474b774a7e52d361bb7",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2027/XGBoost_source_test_predictions.parquet": "48c994af43430f28035a906ef4a46ce6bdfd65c8bfc861bb2f9a278598d7455b",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2027/source_preprocessor.joblib": "1ab0e20cfaa6d64a4ac57c10c51b58b2969863e099ae95efbd49391e21a39969",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2027/source_seed_complete.json": "9bbf2c3fac6e2c94a86af060aecdb171dcc83dddb2d8b4c63e35a55f947fc7bc",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2028/LogisticRegression_model.joblib": "e586884c687edfaf729b5ba40f8939233b644071eae3b7ebcc03c37dfaa500be",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2028/LogisticRegression_source_test_predictions.parquet": "6b6223b70fdffca0b37fbcfb1861784c5aae740fdfd14640ca9810be7132e795",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2028/RandomForest_model.joblib": "7b07c362d3516f378a12be7cda5016132b4bccd916584c106408709579701039",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2028/RandomForest_source_test_predictions.parquet": "b0239176adf9821763d6a4a2cc0ae8cbfbbb83935966be92bf8e2297302e5d25",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2028/XGBoost_model.joblib": "84bb42b6d9b99f554646131709d6add5036142febc82a77b3400a2c2cbfef5ec",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2028/XGBoost_source_test_predictions.parquet": "7f27bd48ca471983c45c1ec778cc8ab19f3bfaf3285c13bf421c12b090127592",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2028/source_preprocessor.joblib": "944517a5e2d638a8cad4ca962295d77ce279d77a91fc2e5d66eb73f56be7cc81",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2028/source_seed_complete.json": "be3a893ad146b2d43275cde33c8ebf1653a2cabc432c84b3f93cc4e88461d1da",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2029/LogisticRegression_model.joblib": "7e2a7d33bfd668f3a49f09336903f0ad9180f35fcd096e3d723e3961b77ea87b",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2029/LogisticRegression_source_test_predictions.parquet": "fc024025e781ee8665d1d1b05a065ab415c8c506025b387c65eb0dfe8bca635c",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2029/RandomForest_model.joblib": "0ffbffb9c6ef33a261f36a362a1711e2186d50e3cdea914063414998e0df6fca",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2029/RandomForest_source_test_predictions.parquet": "0beb0a6b1953d6952a5909e5c02d0347c018b067927471fd3aa74763a3829b94",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2029/XGBoost_model.joblib": "22c4e45e4ab22d7f74d51606841acb6a9ed0df0c9118ca00856a5f370e989a9d",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2029/XGBoost_source_test_predictions.parquet": "af7bc4eb3663c081df317d6c7a41742e75e88202eb8a720cd5c9ec5e034756d1",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2029/source_preprocessor.joblib": "e8a6bd2b6248b61390a09ccd17a2755ea4d783c3eda159973dbd4cc439005fc6",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2029/source_seed_complete.json": "e5ae51af41c6d535086099fdfb3d809573c9d3e7857d7316529baab276e697ad",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2030/LogisticRegression_model.joblib": "ca09a04eb02127cc6e6b37f20576596e09a62960e173cbc05503e9de3f9acd6d",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2030/LogisticRegression_source_test_predictions.parquet": "cad4e575a7b82cbd1ae2a68225ef2f4caee636b658c4088ff9faa49c5add9136",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2030/RandomForest_model.joblib": "fc29b0782ed92e848d7b1b234b321a1cf2fc14e027f8e1629c67d74afe8664a2",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2030/RandomForest_source_test_predictions.parquet": "84bcead6853a32b9c9d5d179470ebf93c0f8911f56393a354b4eabeb675883c5",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2030/XGBoost_model.joblib": "93621a946c728eb5ebdca8e5281d4b66f3f5df5606275432341e1f325024561a",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2030/XGBoost_source_test_predictions.parquet": "865e744dedbc046fa57e3f3d54f594f2be5194980a0979b2f9951cb102c2d47b",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2030/source_preprocessor.joblib": "602e6efcd4370f94a0a4990d27baf2048e1e3fdecb6b3d4d85b5b892190d855f",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2030/source_seed_complete.json": "9db2d996fc408da009b1453dc341dfc95a488273de3fa50351f5bfeb35634416",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2031/LogisticRegression_model.joblib": "77a4ba42d4b4741f7cef3d7bdacb9cf4ec39ebbd063d84383e913ef58a921b10",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2031/LogisticRegression_source_test_predictions.parquet": "5008da91b8743d233aae32c2777834a199a8bd228a8589a333f854dec9c4ac07",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2031/RandomForest_model.joblib": "ea57691b6fbd4604df073eb98eb19f1493fe6e5241cc1ee1a4f586e93f448d9d",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2031/RandomForest_source_test_predictions.parquet": "47723884f82f14f7cab2afbbeebd77c859ead5cc4f9ca7a95fd2107e4f11eded",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2031/XGBoost_model.joblib": "f0e2de65596dbfa0a8c82eff142b339902b7326eea3ed33c029dc7ac4c220da3",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2031/XGBoost_source_test_predictions.parquet": "48a1ae414fffda132daa8521c5f54435a0bfd4e4d9079f7a98ecd275b543a3b4",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2031/source_preprocessor.joblib": "c00722fd7276f3719f8f7a7b835c5085ec7e3923aebddffbb9cf0fca076f6b69",
  "Paper3_Source_Models/NF_ToN_IoT_v3/seed_2031/source_seed_complete.json": "1c04c94afdf80c71fde715b560fbfa556521a527d47d53739a0f01569c150e44",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2027/LogisticRegression_model.joblib": "00497b18e2092a684b3caf5dcb2f114493d3a5785aa0f2d894ce54a3de9592fd",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2027/LogisticRegression_source_test_predictions.parquet": "1c272530a2a34b2e45acada753cbf895da1b8dd37f09b5207196355b8823baa9",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2027/RandomForest_model.joblib": "16b4d2364e8ae0ac87f2ed26a11f65ca9af74c8ad812e4dd3f2b7d80c11759a8",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2027/RandomForest_source_test_predictions.parquet": "800ba609b4ac128c5a69b348e17f25e36058827aa1e2ba15aadd6887f656a7df",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2027/XGBoost_model.joblib": "e60ea2af1e9c12db6873872440a5212ebadd6dd999fceecbbd288ebf5afd5970",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2027/XGBoost_source_test_predictions.parquet": "437a55b35c0dc74a1d72f5fcd3faf14285c726f10177adb6212c431e66c7984c",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2027/source_preprocessor.joblib": "de610333e6c9ecd2b610c8aabf866e339a66a52e6eab37abe65f5e5f65d810c0",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2027/source_seed_complete.json": "f43784d5d2ffcc9456e687b94b0ab0f5d376e9a3ee8fef08fc7e44ea028e3a91",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2028/LogisticRegression_model.joblib": "97fdde086590003138e08c2a99cfe98cc1db66fca44e37e7ec4306e0605bbe19",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2028/LogisticRegression_source_test_predictions.parquet": "ea72b650815ef10c5227d02e6ed16109ddae3f271e899b5581a44f5558de5a7d",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2028/RandomForest_model.joblib": "0f404163ad9d5d613e6ff2a7ee332d373cce978d7309353b9675625881999a1d",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2028/RandomForest_source_test_predictions.parquet": "f6ca1f50980dd636eb20ec3e9ed3d496110c6e5058cea8f9f8dfaa5cd3ca4699",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2028/XGBoost_model.joblib": "9b6352547581ac0ae0206650dfa5021ae8e52d8aba280597beab47f22653c2e1",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2028/XGBoost_source_test_predictions.parquet": "549be77303cbbb94688c5cecda90b6570f60c3a56defedf1cb0e4c0c650d3ef3",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2028/source_preprocessor.joblib": "7af14ab50b615e5ba5ccedabac5d06e490621ce3a40e323a1cd724fcf9ab9af2",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2028/source_seed_complete.json": "508f4680f677c810a8352fefd67b219ef0281fb7aa779e03038907278fbd993e",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2029/LogisticRegression_model.joblib": "ca912b7ce61827961c0bfd1aad2d97cd15fdf4e0cff555d0f0519ec718eb5ca0",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2029/LogisticRegression_source_test_predictions.parquet": "80dca9608863c1d9d33cc0885eb9f12ef99cb8835ae9f2ec48c101a3ced8b533",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2029/RandomForest_model.joblib": "b842e316d837f33022ab62a8721678664f98795a762f1738be7ca911d94ce3c7",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2029/RandomForest_source_test_predictions.parquet": "dbdf29facf94f4e6fb8ac4c21855661d14b7e9148e0afab6c363a34fbeaba11c",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2029/XGBoost_model.joblib": "6111e854990ffde731254de46325d355fc39117e0dcb74e9b4efc2a5a2f6dfee",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2029/XGBoost_source_test_predictions.parquet": "570703f2d6eaad77391887d1a1a9c91ed98a29e409f29e6acf8a5332482a2b98",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2029/source_preprocessor.joblib": "b6c2ba61eda5ab8a7c55fda5caae45880d62fb0e28b72c1e703e495d4614e2f6",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2029/source_seed_complete.json": "1b42986e611763d43ce2bfd53850e4749fb472fc4c1bf3ca2de9e540998fbc45",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2030/LogisticRegression_model.joblib": "1be4d05aa1744c18fe9e3ce0e6e0e69af18decb50f4db3e41a30c89078b06489",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2030/LogisticRegression_source_test_predictions.parquet": "a57de2d1ba9aedeef23f0e9fe488fd03ca1e55558d5fdc8cab809b099409346f",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2030/RandomForest_model.joblib": "3f9fc8431086a72182471c447ca260cad85348f99f201c3a06e0a367c9c50b23",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2030/RandomForest_source_test_predictions.parquet": "2694faa6b94262dff9f2fcebe658830bc7d2ddbeb7b7d9e442c1b49b9801b3f3",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2030/XGBoost_model.joblib": "2cc9cc98e580475da80449fd666ed39b8d4a9bcd97862abbf7ad669c34ea6c7f",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2030/XGBoost_source_test_predictions.parquet": "4bad96c385e8821e8d47c1e657abc15c4277537e0208012b6b77e012405dfa7e",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2030/source_preprocessor.joblib": "a3b0149ba605326cfd2d2b339512d285cafc1796da0ff226a28b087c09420313",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2030/source_seed_complete.json": "3e2ad7549514e3437da60933cd3ef96ef2bbc4d0ebafab05ec7496ce86e3dde9",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2031/LogisticRegression_model.joblib": "5bc903298aa7db93912a712e7e5dc931b376c881cbad19305e74423451e8b9a8",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2031/LogisticRegression_source_test_predictions.parquet": "e815dde022cdd784f50b47722b536024d6c5804a89593d18919a95c9ce180991",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2031/RandomForest_model.joblib": "d85496859efa480e8b1d2f6b52e2db5a957edb98debb4d863389576c58e0b583",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2031/RandomForest_source_test_predictions.parquet": "1194394ba8520080a04604427c97f69a563eb668ba34ea9852702e425a998f9d",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2031/XGBoost_model.joblib": "dc7bb42418d40e48066c334fc8c9c3ce2e8a230923ea94208029b4334e10a50e",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2031/XGBoost_source_test_predictions.parquet": "bbbcc4cc500d6471efacc620e33037de9ef041238e6065b0b142bcf050e24f0c",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2031/source_preprocessor.joblib": "29b1ce721ac1b9e4faa82d9378f3a3f7e57e99197584a01e4b661ab8fc22b0d0",
  "Paper3_Source_Models/NF_UNSW_NB15_v3/seed_2031/source_seed_complete.json": "3ea96faa34557130fb60f501a1b80b7768cddf75edce89b4a48af3005c663d19"
}

RAW_INFO = {
  "NF_BoT_IoT_v3": {
    "raw_zip_name": "NF-BoT-IoT-v3.zip",
    "raw_csv_member": "d509c9db7490cf92_NFV3DATA-A11964_A11964/data/NF-BoT-IoT-v3.csv",
    "raw_rows": 16933808,
    "raw_class_counts": {
      "1": 16881819,
      "0": 51989
    }
  },
  "NF_CSE_CIC_IDS2018_v3": {
    "raw_zip_name": "NF-CSE-CIC-IDS2018-v3.zip",
    "raw_csv_member": "f78acbaa2afe1595_NFV3DATA-A11964_A11964/data/NF-CICIDS2018-v3.csv",
    "raw_rows": 20115529,
    "raw_class_counts": {
      "0": 17514626,
      "1": 2600903
    }
  },
  "NF_ToN_IoT_v3": {
    "raw_zip_name": "NF-ToN-IoT-v3.zip",
    "raw_csv_member": "02934b58528a226b_NFV3DATA-A11964_A11964/data/NF-ToN-IoT-v3.csv",
    "raw_rows": 27520260,
    "raw_class_counts": {
      "0": 16792214,
      "1": 10728046
    }
  },
  "NF_UNSW_NB15_v3": {
    "raw_zip_name": "NF-UNSW-NB15-v3.zip",
    "raw_csv_member": "f7546561558c07c5_NFV3DATA-A11964_A11964/data/NF-UNSW-NB15-v3.csv",
    "raw_rows": 2365424,
    "raw_class_counts": {
      "0": 2237731,
      "1": 127693
    }
  }
}


In [ ]:
%%writefile /content/paper3_natural_prevalence_experiment.py
"""Post hoc unstratified target sampling, fixed-source prediction and evaluation."""
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,zipfile,shutil,platform,gc,warnings
import numpy as np
import pandas as pd
import joblib,sklearn
from sklearn.metrics import average_precision_score,matthews_corrcoef

DATASETS=['NF_UNSW_NB15_v3','NF_BoT_IoT_v3','NF_CSE_CIC_IDS2018_v3','NF_ToN_IoT_v3']
MODELS=['LogisticRegression','RandomForest','XGBoost']
SEEDS=list(range(2027,2032))
META={'row_id','Label','block_id_ms','pseudo_environment_id','actual_environment_id'}
N_TARGET=100000
POOL_SIZE=200000
CHUNK_SIZE=100000
PLAN='''Post hoc natural-capture-proportion target evaluation plan, recorded before new sampling and predictions.
Evaluate all twelve directed transfers, five seeds and three original classifiers (180 jobs). No retraining, target tuning, threshold adjustment or new warning regression.
For each target/seed, take the 200,000 smallest deterministic, label-independent SplitMix64 priorities among valid binary-labelled raw rows. Preserve predictors and labels without class quotas. For each source/seed, exclude rows whose cleaned 49-predictor float64 byte signature matches any original source training, calibration, source-test or environment row. Retain the 100,000 smallest priorities after exclusion. If fewer than 100,000 remain, stop that target/direction rather than silently reduce panel size or change the plan.
Because the oversample contains the lowest priorities in the entire raw target, enough retained rows imply the selected panel is exactly the lowest-priority 100,000 eligible raw rows. No extra quota is imposed on either class. Priorities differ by target and seed, not by classifier; classifiers share a panel. Different sources can exclude different rows. Repeated flows within a target are retained unless their signature occurs in the source; these are flow-level samples, not independent session samples.
The eligible population is the raw benchmark conditional on valid binary labels and source-overlap exclusions. Its class proportion may differ from the complete raw capture. Report complete raw capture proportion, oversample proportion and selected proportion. These are benchmark captures, not a claim about production prevalence. Do not use reweighted balanced summaries as substitutes for new predictions.
Apply unchanged source preprocessing and model-specific calibration thresholds. Replay every original source-test prediction first: binary decisions must match exactly and maximum probability difference must be <=1e-7. Keep the original feature order, float32 prediction transform and float64 overlap signatures. Do not bypass compatibility or integrity failures.
Write separate target feature/label files. Generate predictions using the feature file, write and checksum predictions, then join labels for scoring. Original outcomes are already known: this is a post hoc run, not a historical pre-label confirmatory freeze.
Report per-job n, class counts, observed capture/sample prevalence, precision, attack/benign recall, false-positive rate, accuracy, balanced accuracy, F1, MCC and average precision (stepwise PR-AUC). Undefined ratios remain missing. Summarise means/ranges descriptively; no independent-flow confidence intervals or superiority tests. Pooled job observations are not unique flows. The original balanced experiment remains unchanged.
Checkpoint complete source replays, target pools and individual jobs. Reuse only with matching code/configuration/software/input and output checksums. Interrupted raw scans restart their target, while completed stages are reused. Keep local outputs so a Drive failure can still produce an error-report ZIP.
'''

def sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2**20),b''):h.update(b)
    return h.hexdigest()
def js(p):return json.loads(Path(p).read_text())
def write(p,obj):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);t=p.with_suffix(p.suffix+'.tmp')
    t.write_text(json.dumps(obj,indent=2,allow_nan=False)+'\n');t.replace(p)
def parquet(p,f):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);t=p.with_suffix('.tmp');f.to_parquet(t,index=False);t.replace(p)
def canonical_keys(frame,features):
    a=frame[features].to_numpy(np.float64,copy=True);a[~np.isfinite(a)]=np.nan;a[a==0]=0.;a[np.isnan(a)]=np.nan
    a=np.ascontiguousarray(a.astype('<f8',copy=False))
    return a.view(np.dtype((np.void,8*len(features)))).ravel().copy()
def priorities(target,seed,ids):
    salt=int.from_bytes(hashlib.sha256(f'paper3-natural-v1|{target}|{seed}'.encode()).digest()[:8],'little')
    x=np.asarray(ids,dtype=np.uint64)^np.uint64(salt)
    with np.errstate(over='ignore'):
        x=x+np.uint64(0x9E3779B97F4A7C15);x=(x^(x>>np.uint64(30)))*np.uint64(0xBF58476D1CE4E5B9)
        x=(x^(x>>np.uint64(27)))*np.uint64(0x94D049BB133111EB);x=x^(x>>np.uint64(31))
    return x
def retain(existing,incoming,k):
    f=incoming if existing is None else pd.concat([existing,incoming],ignore_index=True)
    if len(f)>k:f=f.iloc[np.argpartition(f.selection_key.to_numpy(np.uint64),k-1)[:k]]
    return f.reset_index(drop=True)
def select_panel(pool,source_keys,features,n):
    keys=canonical_keys(pool,features);eligible=~np.isin(keys,source_keys)
    f=pool.loc[eligible].sort_values(['selection_key','row_id'],kind='stable')
    if len(f)<n:raise ValueError(f'Oversample insufficient after source exclusion: {len(f)} < {n}. No panel size or quota was changed.')
    selected=f.iloc[:n].sort_values('row_id').reset_index(drop=True)
    if np.isin(canonical_keys(selected,features),source_keys).any():raise ValueError('Final exact source overlap')
    return selected,int((~eligible).sum())
def scores(y,pred,prob):
    y=np.asarray(y,int);pred=np.asarray(pred,int);prob=np.asarray(prob,float)
    tn,fp,fn,tp=map(int,np.bincount(2*y+pred,minlength=4))
    div=lambda a,b:a/b if b else None
    ar=div(tp,tp+fn);br=div(tn,tn+fp);precision=div(tp,tp+fp)
    return {'rows':len(y),'benign_rows':tn+fp,'attack_rows':tp+fn,'attack_fraction':float(y.mean()),
        'tn':tn,'fp':fp,'fn':fn,'tp':tp,'precision':precision,'attack_recall':ar,'benign_recall':br,
        'false_positive_rate':div(fp,tn+fp),'accuracy':div(tp+tn,len(y)),
        'balanced_accuracy':(ar+br)/2 if ar is not None and br is not None else None,
        'f1':div(2*tp,2*tp+fp+fn),'mcc':float(matthews_corrcoef(y,pred)) if (tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)>0 else None,
        'pr_auc_average_precision':float(average_precision_score(y,prob)) if len(np.unique(y))==2 else None,
        'class_support':'BOTH_CLASSES' if len(np.unique(y))==2 else 'ONE_CLASS_ONLY'}
def transformed(prep,f,features,scaled):
    x=prep['imputer'].transform(f[features].to_numpy(np.float32)).astype(np.float32)
    if scaled:x=prep['logistic_regression_scaler'].transform(x).astype(np.float32)
    if not np.isfinite(x).all():raise ValueError('Nonfinite transformed values')
    return x

def run_natural(drive_root,local_root,local_inputs):
    import xgboost,scipy,pyarrow
    drive_root=Path(drive_root);local_root=Path(local_root);local_inputs=Path(local_inputs)
    local_root.mkdir(parents=True,exist_ok=True);local_inputs.mkdir(parents=True,exist_ok=True)
    remote=drive_root/'supplementary_natural_prevalence_20261024/run_v1'
    report={'status':'RUNNING','started_utc':datetime.now(timezone.utc).isoformat(),'completed_jobs':0,'errors':[],'model_load_warnings':[]}
    software={'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'sklearn':sklearn.__version__,'xgboost':xgboost.__version__,'joblib':joblib.__version__,'scipy':scipy.__version__,'pyarrow':pyarrow.__version__}
    config={'software':software,'target_rows':N_TARGET,'pool_rows':POOL_SIZE,'chunk_size':CHUNK_SIZE,'seeds':SEEDS,'datasets':DATASETS,
        'code_sha256':hashlib.sha256(EXPERIMENT_SOURCE.encode()).hexdigest(),'plan_sha256':hashlib.sha256(PLAN.encode()).hexdigest(),
        'raw_info_sha256':hashlib.sha256(json.dumps(RAW_INFO,sort_keys=True).encode()).hexdigest(),
        'pinned_inputs_sha256':hashlib.sha256(json.dumps(PINNED_INPUTS,sort_keys=True).encode()).hexdigest()}
    def upload(rel):
        src=local_root/rel;dst=remote/rel;dst.parent.mkdir(parents=True,exist_ok=True)
        temp=dst.with_suffix(dst.suffix+'.tmp');shutil.copyfile(src,temp)
        if sha(src)!=sha(temp):raise IOError('Checkpoint copy checksum failed: '+str(rel))
        temp.replace(dst)
    def restore(rel):
        p=local_root/rel
        if not p.exists() and (remote/rel).is_file():
            p.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(remote/rel,p)
        return p
    def finish(marker,identity,rels,details):
        record={'identity':identity,'output_hashes':{str(r):sha(local_root/r) for r in rels},'details':details}
        write(local_root/marker,record)
        for r in rels:upload(r)
        upload(marker) # Completion marker copied last.
        return record
    def cached(marker,identity):
        p=restore(marker)
        if not p.exists():return None
        c=js(p)
        if c['identity']!=identity:raise ValueError('Checkpoint identity changed: '+str(marker))
        for r,h in c['output_hashes'].items():
            q=restore(r)
            if not q.is_file() or sha(q)!=h:raise ValueError('Checkpoint output changed: '+r)
        return c
    def stage(rel):
        rel=str(rel);expected=PINNED_INPUTS.get(rel)
        if not expected:raise ValueError('Input has no verified pinned hash: '+rel)
        dst=local_inputs/rel
        if dst.exists() and sha(dst)==expected:return dst
        dst.parent.mkdir(parents=True,exist_ok=True);temp=dst.with_suffix(dst.suffix+'.tmp')
        shutil.copyfile(drive_root/rel,temp)
        if sha(temp)!=expected:raise ValueError('Historical input hash differs: '+rel)
        temp.replace(dst);return dst
    def load_source(source,seed):
        base=f'Paper3_Source_Models/{source}/seed_{seed}'
        meta=js(stage(base+'/source_seed_complete.json'))
        with warnings.catch_warnings(record=True) as ws:
            warnings.simplefilter('always');prep=joblib.load(stage(base+'/source_preprocessor.joblib'))
        report['model_load_warnings']+=list(dict.fromkeys(str(w.message) for w in ws))
        if list(prep['features'])!=features:raise ValueError('Preprocessor feature order mismatch')
        return base,meta,prep
    def load_model(base,meta,source,seed,model):
        with warnings.catch_warnings(record=True) as ws:
            warnings.simplefilter('always');bundle=joblib.load(stage(base+'/'+model+'_model.joblib'))
        report['model_load_warnings']+=list(dict.fromkeys(str(w.message) for w in ws))
        if bundle['dataset']!=source or int(bundle['seed'])!=seed or bundle['model_name']!=model or list(bundle['features'])!=features:raise ValueError('Model identity/schema mismatch')
        scaled=bool(bundle['uses_standard_scaler'])
        if scaled!=(model=='LogisticRegression'):raise ValueError('Unexpected scaler choice')
        threshold=float(bundle['threshold']);info=meta['models'][model]
        if not np.isfinite(threshold) or not 0<=threshold<=1 or abs(threshold-float(info['source_test_metrics']['threshold']))>1e-15:raise ValueError('Threshold changed')
        if not np.array_equal(bundle['model'].classes_,[0,1]):raise ValueError('Unexpected estimator classes')
        return bundle['model'],threshold,scaled
    (local_root/'PLAN.md').write_text(PLAN)
    (local_root/'EXPERIMENT_CODE.py').write_text(EXPERIMENT_SOURCE)
    write(local_root/'PINNED_INPUTS.json',PINNED_INPUTS)
    try:
        if sklearn.__version__!='1.6.1' or xgboost.__version__!='3.4.1':raise ValueError('Run the dependency cell first; original sklearn/XGBoost versions are required')
        lock=restore('run_identity.json')
        if lock.exists() and js(lock)!=config:raise ValueError('Code/config/software differs from existing run; restore original runtime or use an explicitly new run')
        write(lock,config);upload('run_identity.json')
        (local_root/'PLAN.md').write_text(PLAN);(local_root/'EXPERIMENT_CODE.py').write_text(EXPERIMENT_SOURCE)
        write(local_root/'PINNED_INPUTS.json',PINNED_INPUTS)
        for r in ['PLAN.md','EXPERIMENT_CODE.py','PINNED_INPUTS.json']:upload(r)
        features=js(stage('Paper3_Materialized/feature_schema_v1.json'))['predictors_in_order']
        if len(features)!=49:raise ValueError('Expected 49 predictors')
        # Replay all source-test predictions before scanning target archives.
        compatibility=[]
        for source in DATASETS:
            for seed in SEEDS:
                base,meta,prep=load_source(source,seed)
                test=pd.read_parquet(stage(f'Paper3_Sanitized/{source}/seed_{seed}/source_test.parquet'))
                if [c for c in test.columns if c not in META]!=features:raise ValueError('Source feature order differs')
                for model in MODELS:
                    marker=f'compatibility/{source}/seed_{seed}/{model}.json'
                    c=cached(marker,config)
                    if c is None:
                        est,threshold,scaled=load_model(base,meta,source,seed,model)
                        saved=pd.read_parquet(stage(base+'/'+model+'_source_test_predictions.parquet')).set_index('row_id',verify_integrity=True).loc[test.row_id]
                        if len(saved)!=len(test) or not np.array_equal(saved.y_true.to_numpy(),test.Label.to_numpy()):raise ValueError('Source replay labels differ')
                        prob=est.predict_proba(transformed(prep,test,features,scaled))[:,1]
                        delta=float(np.max(np.abs(prob-saved.probability_attack.to_numpy())))
                        if delta>1e-7 or not np.array_equal((prob>=threshold).astype(int),saved.y_pred.to_numpy()):raise ValueError('Source replay failed; no target scan started')
                        c=finish(marker,config,[],{'source':source,'seed':seed,'model':model,'maximum_probability_difference':delta,'binary_predictions_match':True})
                        del est
                    compatibility.append(c['details']);print('Source replay verified:',source,seed,model,flush=True)
                del prep,test;gc.collect()
        write(local_root/'source_replay_checks.json',compatibility);upload('source_replay_checks.json')
        metrics=[];panel_summary=[]
        for target in DATASETS:
            raw=RAW_INFO[target];ziprel=raw['raw_zip_name'];pool_marker=f'pools/{target}/complete.json'
            c=cached(pool_marker,config)
            if c is None:
                archive=stage(ziprel);pools={s:None for s in SEEDS};cutoffs={s:None for s in SEEDS};offset=0;class_counts={0:0,1:0};invalid=0
                print('Scanning unstratified raw target:',target,flush=True)
                with zipfile.ZipFile(archive) as z:
                    with z.open(raw['raw_csv_member']) as stream:
                        for chunk_no,chunk in enumerate(pd.read_csv(stream,usecols=features+['Label'],chunksize=CHUNK_SIZE,low_memory=False),1):
                            y=pd.to_numeric(chunk.Label,errors='coerce').to_numpy();valid=np.isin(y,[0,1]);ids=np.arange(offset,offset+len(chunk),dtype=np.int64)
                            for k in [0,1]:class_counts[k]+=int((y==k).sum())
                            invalid+=int((~valid).sum())
                            for seed in SEEDS:
                                key=priorities(target,seed,ids);mask=valid.copy()
                                if cutoffs[seed] is not None:mask&=key<cutoffs[seed]
                                if mask.any():
                                    inc=chunk.loc[mask,features].copy();inc.insert(0,'row_id',ids[mask]);inc.insert(1,'selection_key',key[mask]);inc['Label']=y[mask].astype('int8')
                                    pools[seed]=retain(pools[seed],inc,POOL_SIZE)
                                    if len(pools[seed])==POOL_SIZE:cutoffs[seed]=pools[seed].selection_key.max()
                            offset+=len(chunk)
                            write(local_root/'progress.json',{'status':'SCANNING_TARGET','target':target,'rows_scanned':offset,'completed_jobs':len(metrics)})
                            if chunk_no%10==0:print(target,format(offset,','),'rows scanned',flush=True)
                if offset!=raw['raw_rows'] or class_counts!={int(k):v for k,v in raw['raw_class_counts'].items()} or invalid:raise ValueError('Raw counts differ from verified metadata scan')
                rels=[]
                for seed,pool in pools.items():
                    if pool is None or len(pool)!=POOL_SIZE or not pool.row_id.is_unique or not pool.selection_key.is_unique:raise ValueError('Incomplete/invalid oversample')
                    # Clean exactly as used for original source-target signature comparisons.
                    pool[features]=pool[features].astype(np.float64).replace([np.inf,-np.inf],np.nan)
                    rel=f'pools/{target}/seed_{seed}.parquet';parquet(local_root/rel,pool.sort_values('selection_key'));rels.append(rel)
                finish(pool_marker,config,rels,{'raw_rows':offset,'class_counts':{str(k):v for k,v in class_counts.items()}})
                del pools,pool;gc.collect()
            for source in DATASETS:
                if source==target:continue
                for seed in SEEDS:
                    prefix=f'jobs/{source}__to__{target}/seed_{seed}'
                    allcached=[cached(prefix+'/'+m+'.json',config) for m in MODELS]
                    if all(x is not None for x in allcached):
                        for x in allcached:metrics.append(x['details']['metrics'])
                        report['completed_jobs']=len(metrics)
                        panel_summary.append(allcached[0]['details']['panel']);print('Reused all three jobs:',source,target,seed,flush=True);continue
                    # Exact byte-signature exclusion uses all four original source partitions.
                    parts=[]
                    for split in ['train','calibration','source_test','environment']:
                        p=stage(f'Paper3_Sanitized/{source}/seed_{seed}/{split}.parquet');sf=pd.read_parquet(p)
                        if [c for c in sf.columns if c not in META]!=features:raise ValueError('Source schema differs')
                        parts.append(canonical_keys(sf,features));del sf
                    source_keys=np.unique(np.concatenate(parts));del parts
                    pool=pd.read_parquet(local_root/f'pools/{target}/seed_{seed}.parquet')
                    selected,excluded=select_panel(pool,source_keys,features,N_TARGET)
                    panel={'source_dataset':source,'target_dataset':target,'seed':seed,'pool_rows':len(pool),'pool_rows_excluded_by_source_signature':excluded,
                        'selected_rows':len(selected),'pool_attack_fraction':float(pool.Label.mean()),'selected_attack_fraction':float(selected.Label.mean()),
                        'raw_capture_attack_fraction':raw['raw_class_counts']['1']/raw['raw_rows'],'source_overlap_rows':0,'class_quotas_used':False}
                    fp=prefix+'/features.parquet';lp=prefix+'/labels.parquet'
                    # On partial resume these deterministic files must reproduce the earlier checkpoint hashes.
                    parquet(local_root/fp,selected[['row_id']+features]);parquet(local_root/lp,selected[['row_id','Label']])
                    panel['feature_sha256']=sha(local_root/fp);panel['label_sha256']=sha(local_root/lp)
                    del source_keys,pool,selected;gc.collect()
                    base,meta,prep=load_source(source,seed)
                    for model,old in zip(MODELS,allcached):
                        marker=prefix+'/'+model+'.json'
                        if old is not None:
                            if old['details']['panel']!=panel:raise ValueError('Rebuilt panel differs from partial checkpoint')
                            metrics.append(old['details']['metrics']);report['completed_jobs']=len(metrics);continue
                        est,threshold,scaled=load_model(base,meta,source,seed,model)
                        feature_frame=pd.read_parquet(local_root/fp)
                        prob=est.predict_proba(transformed(prep,feature_frame,features,scaled))[:,1]
                        if len(prob)!=N_TARGET or not np.isfinite(prob).all() or not ((prob>=0)&(prob<=1)).all():raise ValueError('Invalid target probabilities')
                        pred=(prob>=threshold).astype('int8');prediction=pd.DataFrame({'row_id':feature_frame.row_id,'attack_probability':prob,'predicted_label':pred})
                        pr=prefix+'/'+model+'_predictions.parquet';parquet(local_root/pr,prediction)
                        prediction_sha=sha(local_root/pr) # Predictions written before this new scoring label join.
                        labels=pd.read_parquet(local_root/lp)
                        if not prediction.row_id.equals(labels.row_id):raise ValueError('Target label/prediction alignment failed')
                        score=scores(labels.Label,pred,prob)
                        # Independent boolean count check.
                        yy=labels.Label.to_numpy(int)
                        for k,value in {'tn':int(((yy==0)&(pred==0)).sum()),'fp':int(((yy==0)&(pred==1)).sum()),'fn':int(((yy==1)&(pred==0)).sum()),'tp':int(((yy==1)&(pred==1)).sum())}.items():
                            if score[k]!=value:raise ValueError('Independent confusion calculation differs')
                        score.update({'source_dataset':source,'target_dataset':target,'seed':seed,'model':model,'threshold':threshold,
                            'raw_capture_attack_fraction':panel['raw_capture_attack_fraction'],'prediction_sha256':prediction_sha})
                        finish(marker,config,[fp,lp,pr],{'metrics':score,'panel':panel,'predictions_written_before_new_scoring_join':True})
                        metrics.append(score);report['completed_jobs']=len(metrics)
                        write(local_root/'progress.json',report);print('Completed:',len(metrics),'/ 180 |',source,'to',target,seed,model,flush=True)
                        del est,feature_frame,prediction,labels,prob,pred;gc.collect()
                    panel_summary.append(panel);del prep;gc.collect()
        if len(metrics)!=180 or len(panel_summary)!=60:raise ValueError('Incomplete expected experiment')
        pd.DataFrame(metrics).to_csv(local_root/'natural_prevalence_metrics_180.csv',index=False)
        pd.DataFrame(panel_summary).to_csv(local_root/'panel_sampling_summary_60.csv',index=False)
        numeric=['precision','attack_recall','false_positive_rate','balanced_accuracy','pr_auc_average_precision']
        pd.DataFrame(metrics).groupby(['source_dataset','target_dataset','model'])[numeric].agg(['mean','min','max','count']).to_csv(local_root/'direction_model_descriptive_summary.csv')
        report.update(status='PASS',completed_jobs=180,panels=60,source_model_replays=60,detector_retraining=False,threshold_changes=False,
            natural_capture_sampling=True,production_prevalence_claim=False,completed_utc=datetime.now(timezone.utc).isoformat())
    except Exception as e:
        report.update(status='INCOMPLETE',error_type=type(e).__name__,error=str(e));print('STOPPED:',type(e).__name__,str(e),flush=True)
    write(local_root/'verification_summary.json',report);write(local_root/'progress.json',report)
    # Report generation uses local disk, even if the Drive mount failed.
    files=['PLAN.md','EXPERIMENT_CODE.py','PINNED_INPUTS.json','run_identity.json','source_replay_checks.json','natural_prevalence_metrics_180.csv',
        'panel_sampling_summary_60.csv','direction_model_descriptive_summary.csv','verification_summary.json','progress.json']
    write(local_root/'report_hashes.json',{r:sha(local_root/r) for r in files if (local_root/r).is_file()});files.append('report_hashes.json')
    package=local_root/('natural_prevalence_report_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')+'.zip')
    with zipfile.ZipFile(package,'x',zipfile.ZIP_DEFLATED) as z:
        for r in files:
            if (local_root/r).is_file():z.write(local_root/r,r)
        for p in sorted((local_root/'jobs').rglob('*.json')) if (local_root/'jobs').exists() else []:
            z.write(p,str(p.relative_to(local_root)))
    try:
        for r in files:
            if (local_root/r).is_file():upload(r)
        shutil.copyfile(package,remote/package.name)
        print('Drive results folder:',remote)
    except Exception as e:print('Drive save unavailable. Keep this runtime and download the LOCAL report ZIP:',type(e).__name__,str(e))
    print('Status:',report['status']);print('Download ZIP:',package)
    return package


In [ ]:
# Rerun this cell after reconnecting Drive to resume completed stages.
from pathlib import Path
EXPERIMENT_SOURCE=Path('/content/paper3_natural_prevalence_experiment.py').read_text()
exec(compile(EXPERIMENT_SOURCE,'paper3_natural_prevalence_experiment.py','exec'),globals())
results_zip=run_natural(
    PAPER3_ROOT,
    Path('/content/paper3_natural_prevalence_20261024/run_v1'),
    Path('/content/paper3_natural_inputs_20261024'))
from google.colab import files
files.download(str(results_zip))
